# Data Validation
**This file will validate the data exploration efforts to ensure accurate data handling.**

#### Task 12: independently comparing the record counts from the raw dataset to the prepared dataset, taking into consideration the removal of duplicated rows.

Utilizing 01_data_explortation prepared dataset.

In [23]:
import pandas as pd

#loading prepared dataset
df_prepared = pd.read_csv("data/restaurants_prepared.csv",
                          dtype={"ZIPCODE": str, "Community Board": str, "Council District": str,
                                 "Census Tract": str, "BIN": str, "BBL": str},
                          parse_dates=["INSPECTION DATE", "GRADE DATE", "RECORD DATE"],
                          low_memory=False)

# loading raw dataset
df_raw = pd.read_csv("data/restaurant_inspections.csv")

#loading excluded records
df_exclusion_log = pd.read_csv("handoff/duplicate_exclusion_log.csv")

Record counts from _01_data_exploration_.

In [24]:
print("Raw records:", len(df_raw))
print("Prepared records:", len(df_prepared))

print("Expected prepared records:",
      len(df_raw) - len(df_exclusion_log))

Raw records: 296012
Prepared records: 295843
Expected prepared records: 295843


Checking source identifiers (number of source IDs should equal number of records in prepared dataset).

In [25]:
print("Unique source IDs:", df_prepared["SOURCE_ROW_ID"].nunique())

print("Missing source IDs:", df_prepared["SOURCE_ROW_ID"].isna().sum())

Unique source IDs: 295843
Missing source IDs: 0


#### Task 14: Validating Categorical Values

This code looks at the categorial variables for missing values and data inconsistencies.

Task 13 is located in _03_data_dictionary_.

In [38]:
#prelimiary check for unique values
categorical_col = ["BORO", "CUISINE DESCRIPTION", "ACTION", "INSPECTION TYPE",
                   "GRADE", "CRITICAL FLAG", "VIOLATION CODE", "VIOLATION DESCRIPTION"]

for col in categorical_col:
    print("\n")
    print(df_prepared[col].value_counts(dropna=False))



BORO
Manhattan        109608
Brooklyn          74873
Queens            74192
Bronx             27226
Staten Island      9593
0                   351
Name: count, dtype: int64


CUISINE DESCRIPTION
American              44081
Chinese               29107
Coffee/Tea            21582
Pizza                 17033
Latin American        14854
                      ...  
Nuts/Confectionary       32
Czech                    19
Polynesian               10
Haute Cuisine             8
Chimichurri               4
Name: count, Length: 91, dtype: int64


ACTION
Violations were cited in the following area(s).                                                                                       275767
Establishment Closed by DOHMH. Violations were cited in the following area(s) and those requiring immediate action were addressed.     10839
NaN                                                                                                                                     3714
No violations were reco

Preliminary checks show some missing values, but the text data appears consistent. Each column will be checked for amount of nulls.

In [ ]:
#check for null values
categorical_col = ["BORO", "CUISINE DESCRIPTION", "ACTION", "INSPECTION TYPE",
                   "GRADE", "CRITICAL FLAG", "VIOLATION CODE", "VIOLATION DESCRIPTION"]

for col in categorical_col:
    print(f"\nAmount of {col} nulls:")
    print(df_prepared[col].isna().sum())


Amount of BORO nulls:
0

Amount of CUISINE DESCRIPTION nulls:
3794

Amount of ACTION nulls:
3714

Amount of INSPECTION TYPE nulls:
3714

Amount of GRADE nulls:
150537

Amount of CRITICAL FLAG nulls:
0

Amount of VIOLATION CODE nulls:
6444

Amount of VIOLATION DESCRIPTION nulls:
6444


#### Task 15: Investigate missing-value patterns

This code aims to understand the relationships between columns with missing values.

Will first separate records based on the already created _AWAITING_INSPECTION_ flag, and analyze the relationships between the records inspection status and their action, inspection type, and grade statuses.

In [ ]:
#df for records awaiting inspection
df_awaiting = df_prepared[df_prepared["AWAITING_INSPECTION"] == True]

#df for records that have already received an inspection
df_inspected = df_prepared[df_prepared["AWAITING_INSPECTION"] == False]

inspected_cols = [
    "ACTION",
    "INSPECTION TYPE",
    "GRADE",
    "VIOLATION CODE",
    "VIOLATION DESCRIPTION",
    "CUISINE DESCRIPTION"
]

print(df_inspected[inspected_cols].isna().sum())

ACTION                        0
INSPECTION TYPE               0
GRADE                    146823
VIOLATION CODE             2730
VIOLATION DESCRIPTION      2730
CUISINE DESCRIPTION          80
dtype: int64


Examining the relationship between all categorical variables with nulls and their inspection status.

In [76]:
def count_missing(x):
    return x.isna().sum()

In [77]:
df_prepared.groupby("AWAITING_INSPECTION")[["ACTION", "INSPECTION TYPE", "CUISINE DESCRIPTION", 
                                            "GRADE", "VIOLATION CODE", "VIOLATION DESCRIPTION"]].apply(
    count_missing
)

,ACTION,INSPECTION TYPE,CUISINE DESCRIPTION,GRADE,VIOLATION CODE,VIOLATION DESCRIPTION
AWAITING_INSPECTION,,,,,,
False,0,0,80,146823,2730,2730
True,3714,3714,3714,3714,3714,3714


Findings:
- Records that lack _ACTION_, _INSPECTION TYPE_, _CUISINE DESCRIPTION_, __AND__ _GRADE_ values are those that are awaiting an inspection (or have the placeholder 1900-01-01 as their inspection date).
- Of records that have been inspected, 80 lack a _CUISINE DESCRIPTION_, AND 146,823 lack a GRADE.

How many inspected records lacking a violation code also lack a violation description?

In [43]:
pd.crosstab(df_inspected["VIOLATION CODE"].isna(),
            df_inspected["VIOLATION DESCRIPTION"].isna(),
            rownames = ["Code Missing"],
            colnames = ["Description Missing"],
            margins = True)

Description Missing,False,True,All
Code Missing,,,
False,289399,0,289399
True,0,2730,2730
All,289399,2730,292129


Findings:
- Records missing a violation description __and__ missing a code total 2730.

How many inspected records are missing violation details and a score?

In [44]:
pd.crosstab(df_inspected["SCORE"].isna(),
            df_inspected["VIOLATION CODE"].isna(),
            rownames = ["Score Missing"],
            colnames = ["Code Missing"],
            margins = True)

Code Missing,False,True,All
Score Missing,,,
False,276811,1812,278623
True,12588,918,13506
All,289399,2730,292129


Findings:
- Records missing __only__ a violation code (and thus description) total 1,812.
- Records missing __only__ a score total 12,588.
- Records missing __both__ a score and violation code total 918.

#### Task 16: Missing value handling

Of the categorical columns examined, the following contain null values.
- ACTION
- INSPECTION TYPE
- GRADE
- VIOLATION CODE
- VIOLATION DESCRIPTION
- CUISINE DESCRIPTION

The only variable with missing values to be handled for now is the _CUISINE DESCRIPTION_ variable. All other variables that have missing rows will be left as null values; using a placeholder may distort the analysis.

Missing cuisine descriptions will be filled with an "Unknown" placeholder. The cuisine descriptions currently do not appear to answer the main question of:
- _What factors are associated with restaurant food safety performance in NYC, and can historical inspection records help identify restaurants at greater risk of poor inspection outcomes?_

In [133]:
df_prepared["CUISINE DESCRIPTION"] = (df_prepared["CUISINE DESCRIPTION"].fillna("Unknown"))
df_prepared.to_csv("restaurants_prepared.csv",index=False)

In [134]:
print("Missing cuisine descriptions:",
      df_prepared["CUISINE DESCRIPTION"].isna().sum())

print("Unknown cuisine descriptions:",
      (df_prepared["CUISINE DESCRIPTION"] == "Unknown").sum())

Missing cuisine descriptions: 0
Unknown cuisine descriptions: 3794


A csv file is created detailing the missing value counts of the selected categorical variables, along with the imputation decisions.

In [ ]:

value_amounts = {
    "BORO": 0, 
    "CUISINE DESCRIPTION": 3794, 
    "ACTION": 3714, 
    "INSPECTION TYPE": 3714,
    "GRADE": 150537, 
    "CRITICAL FLAG": 0, 
    "VIOLATION CODE": 6444, 
    "VIOLATION DESCRIPTION": 6444
}

imputation_description = {
    "BORO": "", 
    "CUISINE DESCRIPTION": "Change NaN to 'Unknown'. This should not distort future analysis.", 
    "ACTION": "", 
    "INSPECTION TYPE": "",
    "GRADE": "Left as missing. The grade can be subject to change, and re-inspection determines the grade given.", 
    "CRITICAL FLAG": "", 
    "VIOLATION CODE": "", 
    "VIOLATION DESCRIPTION": ""
}

missing = {
    "Column Name": categorical_col,
    "Number of Missing Values": [value_amounts[col] for col in categorical_col],
    "Imputation Decision": [imputation_description[col] for col in categorical_col]
}

df_missing = pd.DataFrame(missing)
df_missing.to_csv("handoff/categorical_missing.csv", index=False)
print("DataFrame successfully saved to categorical_missing.csv!")

DataFrame successfully saved to categorical_missing.csv!


#### Task 17: Validate Inspection Scores

Inspection scores are examined for extreme outliers. 

Resources on Scoring: 
- https://a816-health.nyc.gov/ABCEatsRestaurants/#!/faq 
- https://helpnewyork.com/nyc-restaurant-health-inspection-guide/
- https://www.nyc.gov/assets/doh/downloads/pdf/about/healthcode/health-code-chapter23.pdf


The ABCEats resource states:
- "A" grade: 0 to 13 points for sanitary violations
- "B" grade: 14 to 27 points for sanitary violations
- "C" grade: 28 or more points for sanitary violations

In [69]:
print("Score Stats:",
      df_prepared["SCORE"].describe())

print("\nMissing scores:",
      df_prepared["SCORE"].isna().sum())

print("\nZeros:",
      (df_prepared["SCORE"] == 0).sum())

Score Stats: count    278623.000000
mean         25.630741
std          19.157615
min           0.000000
25%          12.000000
50%          22.000000
75%          33.000000
max         214.000000
Name: SCORE, dtype: float64

Missing scores: 17220

Zeros: 3008


Most scores fall close to the 0-28 range, with a max score of 214. A max score of 214 seems concerning, but may not be wrong. A score of 28+ signifies a low grade, so it is possible that a score of 214 was the result of multiple sanitary violations. Scores of zero indicate that the inspection found no violations.

There are about a quarter of records above 33, but with no ceiling threshold on the scores, it may be the case that these higher scores are also just as valid as the lower ranges. 

In [61]:
score_214 = df_prepared[df_prepared["SCORE"] == 214]

print(score_214)

        SOURCE_ROW_ID     CAMIS           DBA       BORO BUILDING  \
3874             3877  50182863  TAPAS ON LEX  Manhattan     1643   
24946           24964  50182863  TAPAS ON LEX  Manhattan     1643   
29097           29118  50182863  TAPAS ON LEX  Manhattan     1643   
31570           31594  50182863  TAPAS ON LEX  Manhattan     1643   
31760           31784  50182863  TAPAS ON LEX  Manhattan     1643   
50472           50505  50182863  TAPAS ON LEX  Manhattan     1643   
61638           61679  50182863  TAPAS ON LEX  Manhattan     1643   
95014           95079  50182863  TAPAS ON LEX  Manhattan     1643   
96841           96906  50182863  TAPAS ON LEX  Manhattan     1643   
186726         186838  50182863  TAPAS ON LEX  Manhattan     1643   
214799         214920  50182863  TAPAS ON LEX  Manhattan     1643   
226386         226517  50182863  TAPAS ON LEX  Manhattan     1643   
266172         266333  50182863  TAPAS ON LEX  Manhattan     1643   
266891         267052  50182863  T

In [65]:
df_prepared.loc[df_prepared["SCORE"] == 214, "VIOLATION DESCRIPTION"]

3874      Evidence of rats or live rats in establishment...
24946     No approved written standard operating procedu...
29097     Non-food contact surface or equipment made of ...
31570     Contract with a pest management professional n...
31760     Food contact surface, refillable, reusable con...
50472     Food, supplies, or equipment not protected fro...
61638     Accurate thermometer not provided or properly ...
95014     Lighting inadequate; permanent lighting not pr...
96841     Dishwashing and ware washing: Cleaning and san...
186726    Anti-siphonage or back-flow prevention device ...
214799    Insufficient or no hot holding, cold storage o...
226386    Establishment is not free of harborage or cond...
266172    Food contact surface not properly washed, rins...
266891    Properly scaled and calibrated thermometer or ...
270202    Wash hands sign not posted near or above hand ...
280571    No hand washing facility in or adjacent to toi...
284498    Food preparation area, food st

In [66]:
df_prepared.loc[df_prepared["SCORE"] == 214, "VIOLATION CODE"]

3874        04K
24946       05H
29097       10F
31570     28-06
31760       05C
50472       06C
61638       10E
95014       10C
96841       10G
186726      10B
214799      05F
226386      08A
266172      06D
266891      04J
270202      09E
280571      05D
284498      04F
291161      10A
Name: VIOLATION CODE, dtype: object

It appears that the inspection records with the highest score of 214 come from the same establishment. It seems entirely plausible that the score of 214 is sensible, as this establishment has multiple records of violations. The NYC documentation states that scores can exceed 28 with no limit, so higher values will be left alone for now.

#### Task 18: Validate Grades and Grade Statuses

In this portion, grades are validated and examined in-depth.

In [73]:
#count of records per grade
print(df_prepared["GRADE"].value_counts(dropna = False))

GRADE
NaN    150537
A       98179
B       18084
C       13230
N       10280
Z        4786
P         747
Name: count, dtype: int64


Checking the amount of grades and their corresponding grade dates to understand discrepancies in the whole grade identity.

In [ ]:
# Checking the grade and grade-date combinations
grade_date_check = pd.crosstab(
    df_prepared["GRADE"].fillna("MISSING"),
    df_prepared["GRADE DATE"].isna(),
    rownames=["Grade / Status"],
    colnames=["Grade Date Missing"],
    margins = True
)

grade_date_check

Grade Date Missing,False,True,All
Grade / Status,,,
A,98179,0,98179
B,18084,0,18084
C,13230,0,13230
MISSING,0,150537,150537
N,10,10270,10280
P,747,0,747
Z,4786,0,4786
All,135036,160807,295843


Here a temporary column called _GRADE_CATEGORY_ is created to notate which records have official letter grades, temporary grades, or are missing grades. This does not modify the original _GRADE_ column, nor does it modify the prepared dataset.

_GRADE_CATEGORY_ is a temporary column only stored in the df_prepared dataframe from this file. This is solely to provide some understanding as to the proportion of establishments assigned an official grade from the inspection records, versus those that either have a temporary grade status or are missing one entirely.

In [98]:
letter_grades = ["A","B","C"]
other_grade_statuses = ["N", "P", "Z"]

df_prepared["GRADE_CATEGORY"] = df_prepared["GRADE"].apply(
    lambda x: (
        "Official Letter Grade" if x in letter_grades
        else "Temporary Grade Status" if x in other_grade_statuses
        else "Missing" if pd.isna(x)
        else "Other"
    )
)

print(df_prepared["GRADE_CATEGORY"])

0                       Missing
1         Official Letter Grade
2                       Missing
3                       Missing
4                       Missing
                  ...          
295838                  Missing
295839    Official Letter Grade
295840                  Missing
295841    Official Letter Grade
295842    Official Letter Grade
Name: GRADE_CATEGORY, Length: 295843, dtype: object


How many inspected records lacking a score also lack a grade?

In [ ]:
pd.crosstab(df_inspected["SCORE"].isna(),
            df_inspected["GRADE"].isna(),
            rownames = ["Score Missing"],
            colnames = ["Grade Missing"],
            margins = True)

Grade Missing,False,True,All
Score Missing,,,
False,145295,133328,278623
True,11,13495,13506
All,145306,146823,292129


Findings:
- Records missing __both__ grades and scores total 13,495.
- Records missing __only__ grades total 133,328.
- Records missing __only__ scores total 11.

What are the inspection types, actions, and scores attached to the missing grades?

In [105]:
# Investigate missing grades among inspected records
missing_grade_records = df_prepared[
    (df_prepared["AWAITING_INSPECTION"] == False) &
    (df_prepared["GRADE"].isna())
]

# Examine missing grades by inspection type
print(
    missing_grade_records["INSPECTION TYPE"].value_counts(dropna = False)
)

INSPECTION TYPE
Cycle Inspection / Initial Inspection                            97097
Pre-permit (Operational) / Initial Inspection                    24019
Administrative Miscellaneous / Initial Inspection                 9590
Pre-permit (Non-operational) / Initial Inspection                 2521
Cycle Inspection / Re-inspection                                  2263
Pre-permit (Operational) / Compliance Inspection                  1820
Administrative Miscellaneous / Re-inspection                      1766
Cycle Inspection / Compliance Inspection                          1510
Cycle Inspection / Reopening Inspection                           1324
Inter-Agency Task Force / Initial Inspection                      1123
Smoke-Free Air Act / Initial Inspection                            647
Pre-permit (Operational) / Reopening Inspection                    551
Pre-permit (Operational) / Re-inspection                           505
Calorie Posting / Initial Inspection                         

In [106]:
# Examine missing grades by inspection action
print(
    missing_grade_records["ACTION"].value_counts(dropna = False)
)
print("\n")

# Examine missing grades by score availability
print(
    missing_grade_records["SCORE"].isna().value_counts()
)

ACTION
Violations were cited in the following area(s).                                                                                       132858
Establishment Closed by DOHMH. Violations were cited in the following area(s) and those requiring immediate action were addressed.     10320
No violations were recorded at the time of this inspection.                                                                             2202
Establishment re-closed by DOHMH.                                                                                                       1254
Establishment re-opened by DOHMH.                                                                                                        189
Name: count, dtype: int64


SCORE
False    133328
True      13495
Name: count, dtype: int64


#### Task 19: Validate Violation Information

Here, violation codes/descriptions are cross-checked alongside critical flags and inspection actions.

We look at the codes and their associated unique descriptions. Many codes have multiple descriptions associated with them. The https://www.nyc.gov/assets/doh/downloads/pdf/about/healthcode/health-code-chapter23.pdf describes that a violation can have multiple conditions (thus having multiple descriptions per one code). 

In [109]:
#codes with multiple descriptions
codes_descriptions_check = (
    df_prepared.groupby("VIOLATION CODE", dropna=False)
    ["VIOLATION DESCRIPTION"].nunique(dropna = True)
    .reset_index(name = "unique_descriptions")
)

# Isolate codes with more than one description
multiple_descriptions = codes_descriptions_check[
    codes_descriptions_check["unique_descriptions"] > 1
]

print(multiple_descriptions)

    VIOLATION CODE  unique_descriptions
0              02A                    3
1              02B                    3
2              02C                    3
5              02G                    3
6              02H                    3
7              02I                    2
8              03A                    3
9              03B                    2
11             03D                    2
12             03E                    3
16             04A                    2
18             04C                    2
19             04D                    2
20             04E                    2
21             04F                    3
24             04J                    2
25             04K                    3
26             04L                    2
27             04M                    3
28             04N                    4
29             04O                    3
31             05A                    2
32             05B                    3
33             05C                    4


Detailed descriptions for the codes:

In [110]:
#descriptions displayed with corresponding codes
details = (
    df_prepared.groupby(
        ["VIOLATION CODE", "VIOLATION DESCRIPTION"],
        dropna = False
    ).size().reset_index(name = "record_count")
)

details[
    details["VIOLATION CODE"]
    .isin(multiple_descriptions["VIOLATION CODE"])
].sort_values("VIOLATION CODE")

,VIOLATION CODE,VIOLATION DESCRIPTION,record_count
0,02A,Food not cooked to required minimum temperature.,6
1,02A,Time/Temperature Control for Safety (TCS) food...,214
2,02A,Time/Temperature Control for Safety (TCS) food...,26
3,02B,Hot TCS food item not held at or above 140 °F.,15350
4,02B,Hot food item not held at or above 140º F.,218
...,...,...,...
220,20-06,Current letter grade or Grade Pending card not...,2191
235,28-05,Food adulterated or misbranded. Adulterated o...,37
236,28-05,Food adulterated or misbranded. Adulterated or...,493
237,28-06,Contract with a pest management professional n...,71


Here, we examine unique critical flags present in the violations.

In [111]:
# Count unique critical flag values per violation code
critical_flag_check = (
    df_prepared.groupby("VIOLATION CODE", dropna=False)
    ["CRITICAL FLAG"]
    .nunique(dropna=False)
    .reset_index(name="unique_critical_flags")
)

# Identify codes with multiple critical flag values
inconsistent_critical_flags = critical_flag_check[
    critical_flag_check["unique_critical_flags"] > 1
]

print(inconsistent_critical_flags)

   VIOLATION CODE  unique_critical_flags
51            09A                      2


We see that 09A is the only violation code to have both _NOT CRITICAL_ and _CRITICAL_ critical flags. We will retrieve the description to better understand why.

In [ ]:
#Retrival of the violation code and critial flag combinations.
critical_flag_details = (
    df_prepared.groupby(
        ["VIOLATION CODE", "VIOLATION DESCRIPTION", "CRITICAL FLAG"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

critical_flag_details[
    critical_flag_details["VIOLATION CODE"].isin(
        inconsistent_critical_flags["VIOLATION CODE"]
    )
]

,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,record_count
115,09A,Canned food product observed dented and not se...,Not Critical,16
116,09A,Canned food product observed severly dented.,Not Critical,1
117,09A,"Swollen, leaking, rusted or otherwise damaged ...",Critical,280
118,09A,"Swollen, leaking, rusted or otherwise damaged ...",Not Critical,586


It appears that there are four usual descriptions associated with the violation 09A. It is possible that over time the description of what consitutes an 09A violation has changed, and thus some 09A violations are outdated.

In [ ]:
#Checking year the assigned violations
df_09a = df_prepared[df_prepared["VIOLATION CODE"] == "09A"].copy()

df_09a["YEAR"] = pd.to_datetime(df_09a["INSPECTION DATE"]).dt.year

pd.crosstab(df_09a["YEAR"],
            df_09a["CRITICAL FLAG"])

CRITICAL FLAG,Critical,Not Critical
YEAR,,
2007,0,1
2017,0,2
2018,0,2
2019,0,2
2020,0,2
2021,0,1
2022,0,22
2023,30,86
2024,81,99


Here, we check to see how the action descriptions correlate with the critical flags.

In [87]:
pd.crosstab(df_prepared["ACTION"],
            df_prepared["CRITICAL FLAG"],
            dropna = False)

CRITICAL FLAG,Critical,Not Applicable,Not Critical
ACTION,,,
Establishment Closed by DOHMH. Violations were cited in the following area(s) and those requiring immediate action were addressed.,6941,5,3893
Establishment re-closed by DOHMH.,670,1,583
Establishment re-opened by DOHMH.,235,283,1121
No violations were recorded at the time of this inspection.,0,2444,186
Violations were cited in the following area(s).,147351,1812,126604
NaN,0,3714,0


Now we look to see that combinations of actions and violation information. This gives us some background on the relationships between actions, violation codes/descriptions, and critical flags.

In [113]:
# Examine combinations of inspection action and violation information
action_violation_check = (
    df_prepared.groupby(
        ["ACTION", "VIOLATION CODE", "VIOLATION DESCRIPTION",
         "CRITICAL FLAG"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

action_violation_check.sort_values(
    ["VIOLATION CODE", "ACTION"]
)

,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,record_count
0,Establishment Closed by DOHMH. Violations were...,02A,Time/Temperature Control for Safety (TCS) food...,Critical,10
1,Establishment Closed by DOHMH. Violations were...,02A,Time/Temperature Control for Safety (TCS) food...,Critical,1
277,Violations were cited in the following area(s).,02A,Food not cooked to required minimum temperature.,Critical,6
278,Violations were cited in the following area(s).,02A,Time/Temperature Control for Safety (TCS) food...,Critical,204
279,Violations were cited in the following area(s).,02A,Time/Temperature Control for Safety (TCS) food...,Critical,25
...,...,...,...,...,...
193,Establishment re-closed by DOHMH.,NaN,NaN,Not Applicable,1
252,Establishment re-opened by DOHMH.,NaN,NaN,Not Applicable,283
276,No violations were recorded at the time of thi...,NaN,NaN,Not Applicable,2439
518,Violations were cited in the following area(s).,NaN,NaN,Not Applicable,5


Finally, we create a csv file that holds some basic summary information regarding violations.

In [119]:
violation_validation_summary = pd.DataFrame({
    "Validation Check": [
        "Violation codes with multiple descriptions",
        "Violation codes with multiple critical flags",
        "Records missing violation codes",
        "Records missing violation descriptions",
        "Records missing both violation codes and descriptions"
    ],
    "Records / Codes Flagged": [
        len(multiple_descriptions),
        len(inconsistent_critical_flags),
        df_prepared["VIOLATION CODE"].isna().sum(),
        df_prepared["VIOLATION DESCRIPTION"].isna().sum(),
        (
            df_prepared["VIOLATION CODE"].isna() &
            df_prepared["VIOLATION DESCRIPTION"].isna()
        ).sum()
    ]
})

violation_validation_summary.to_csv(
    "handoff/violation_validation_summary.csv",
    index=False
)

#### Task 20: Final Cross-field consistency check

In [123]:
real_rows = df_prepared[
    df_prepared["AWAITING_INSPECTION"] == False
].copy()

inspection_key = [
    "CAMIS",
    "INSPECTION DATE",
    "INSPECTION TYPE"
]

#columns checked are those from Task 18 and 19, along with some other numeric columns
check_cols = [
    "SCORE", 
    "ACTION", 
    "GRADE",
    "GRADE DATE", 
    "CRITICAL FLAG", 
    "VIOLATION CODE", 
    "VIOLATION DESCRIPTION",
]

grouped_records = real_rows.groupby(inspection_key, dropna = False)
conflicting_records = []

for col in check_cols:
    conflicting_groups = grouped_records[col].nunique(dropna=False)
    conflicting_records.append({"Column": col, 
                                "Conflicting groups": (conflicting_groups > 1).sum()
                                })

conflict_summary = pd.DataFrame(conflicting_records)

print(conflict_summary)

                  Column  Conflicting groups
0                  SCORE                   0
1                 ACTION                   0
2                  GRADE                   0
3             GRADE DATE                   0
4          CRITICAL FLAG               59740
5         VIOLATION CODE               73725
6  VIOLATION DESCRIPTION               73725


The columns of _SCORE_, _ACTION_, _GRADE_, and _GRADE_DATE_ all show no conflicts. 

The last three columns showcase many conflicts, but a single establishment may have multiple critical flags, violation codes, and violation descriptions for the same inspection. Historical changes may also contribute to explaining some of these inconsistencies.

In [131]:
violation_cols = [
    "CRITICAL FLAG",
    "VIOLATION CODE",
    "VIOLATION DESCRIPTION"
]

# Group by violation code
violation_grouped = real_rows.groupby("VIOLATION CODE",dropna=False)

violation_consistency = pd.DataFrame({
    "Multiple Descriptions": violation_grouped["VIOLATION DESCRIPTION"].nunique(dropna=False) > 1,
    "Multiple Critical Flags": violation_grouped["CRITICAL FLAG"].nunique(dropna=False) > 1
})

print(violation_consistency.sum())

Multiple Descriptions      59
Multiple Critical Flags     1
dtype: int64


Here, we see that 59 violation codes have more than one unique description (as seen in 09A in Task 19). We also know that the violation code 09A has both critical and non-critical flags from Task 19. 